In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('Past In and Out.csv')

In [4]:
from datetime import datetime
def Milli(X):
    given_date_time = X
    epoch = datetime(1970, 1, 1)
    time_difference = given_date_time - epoch
    total_seconds = time_difference.total_seconds()
    milliseconds = int(total_seconds * 1000)
    return milliseconds

def Date_Time(X):
    seconds = X / 1000
    date_time = datetime.fromtimestamp(seconds)
    date_time1= date_time.strftime('%Y-%m-%d %H:%M:%S')
    return date_time1

In [5]:
from datetime import datetime
import pandas as pd
IN_Time=[]
OUT_Time=[]

for i in range(df.shape[0]):
    x1 = datetime.strptime(df['IN_TIME'][i], '%d-%m-%y %H:%M:%S')
    x1m = Milli(x1)
    IN_Time.append(x1m)
    y1 = datetime.strptime(df['OUT_TIME'][i], '%d-%m-%y %H:%M:%S')
    y1m = Milli(y1)
    IN_Time.append(x1m)
    OUT_Time.append(y1m)
In = pd.DataFrame(IN_Time)
Out = pd.DataFrame(OUT_Time)


In [6]:
df.drop(columns='REF_ID',axis=1,inplace=True)
df.drop(columns='VALIDITY',axis=1,inplace=True)
df.drop(columns='CON_NUM',axis=1,inplace=True)
df.drop(columns='IN_TIME',axis=1,inplace=True)
df.drop(columns='OUT_TIME',axis=1,inplace=True)

In [7]:
df['In_time'] = In
df['Out_time'] = Out

In [10]:
df.dropna(axis=0,inplace=True)

In [11]:
df.reset_index(inplace=True)

In [12]:
df1=pd.get_dummies(data=df['STATUS'],drop_first=True)
df['L']=df1

In [13]:
df.drop(columns='STATUS',axis=1,inplace=True)
df.drop(columns='index',axis=1,inplace=True)

,CON_SIZE,In_time,Out_time,L
0,40.0,1668216594000,1675732992000,0
1,40.0,1668216594000,1675366422000,0
2,20.0,1669124439000,1676005749000,0
3,20.0,1669124439000,1677572528000,0
4,20.0,1670173982000,1681752196000,0
...,...,...,...,...
24524,40.0,1678738917000,1682467275000,1
24525,40.0,1678738917000,1682478340000,1
24526,20.0,1678738917000,1682474490000,1
24527,40.0,1678740230000,1682488202000,0


In [14]:
Time_diff = []
for i in range(df.shape[0]):
    Time_diff.append(df['Out_time'][i]-df['In_time'][i])
    
df['Time_Diff'] = Time_diff
def find_outliers(df,col):
    q1=df[col].quantile(0.25)
    q3=df[col].quantile(0.75)
    IQR=q3-q1
    lv=q1-1.5*IQR
    hv=q3+1.5*IQR
    df=df[(df[col]<=hv) | (df[col]>=lv)]
    return df
find_outliers(df,'Time_Diff')

,CON_SIZE,In_time,Out_time,L,Time_Diff
0,40.0,1668216594000,1675732992000,0,7516398000
1,40.0,1668216594000,1675366422000,0,7149828000
2,20.0,1669124439000,1676005749000,0,6881310000
3,20.0,1669124439000,1677572528000,0,8448089000
4,20.0,1670173982000,1681752196000,0,11578214000
...,...,...,...,...,...
24524,40.0,1678738917000,1682467275000,1,3728358000
24525,40.0,1678738917000,1682478340000,1,3739423000
24526,20.0,1678738917000,1682474490000,1,3735573000
24527,40.0,1678740230000,1682488202000,0,3747972000


In [15]:
df.drop(columns='Time_Diff',axis=1,inplace=True)

In [17]:
x=df.drop('Out_time',axis=1)
y=df['Out_time']
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=1)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(19623, 3)
(4906, 3)
(19623,)
(4906,)


,CON_SIZE,In_time,Out_time,L
0,40.0,1668216594000,1675732992000,0
1,40.0,1668216594000,1675366422000,0
2,20.0,1669124439000,1676005749000,0
3,20.0,1669124439000,1677572528000,0
4,20.0,1670173982000,1681752196000,0
...,...,...,...,...
24524,40.0,1678738917000,1682467275000,1
24525,40.0,1678738917000,1682478340000,1
24526,20.0,1678738917000,1682474490000,1
24527,40.0,1678740230000,1682488202000,0


In [18]:
import numpy 
from sklearn.linear_model import LinearRegression
lr=LinearRegression()
lr.fit(x_train,y_train)

print(lr.coef_)
print(lr.intercept_)

[2.58619498e+06 1.84151937e+00 1.46778791e+08]
-1409148897269.0027


,CON_SIZE,In_time,Out_time,L
0,40.0,1668216594000,1675732992000,0
1,40.0,1668216594000,1675366422000,0
2,20.0,1669124439000,1676005749000,0
3,20.0,1669124439000,1677572528000,0
4,20.0,1670173982000,1681752196000,0
...,...,...,...,...
24524,40.0,1678738917000,1682467275000,1
24525,40.0,1678738917000,1682478340000,1
24526,20.0,1678738917000,1682474490000,1
24527,40.0,1678740230000,1682488202000,0


In [19]:
y_pred=lr.predict(x_test)
print(y_pred)
print(y_test)

[1.68175294e+12 1.68044246e+12 1.68091108e+12 ... 1.68220033e+12
 1.67711545e+12 1.67787852e+12]
20997    1681911443000
12655    1679226166000
19424    1681999096000
16029    1681056131000
9602     1678633379000
             ...      
3244     1675876990000
3877     1676609170000
23117    1682557658000
7149     1677308338000
5655     1677050680000
Name: Out_time, Length: 4906, dtype: int64


,CON_SIZE,In_time,Out_time,L
0,40.0,1668216594000,1675732992000,0
1,40.0,1668216594000,1675366422000,0
2,20.0,1669124439000,1676005749000,0
3,20.0,1669124439000,1677572528000,0
4,20.0,1670173982000,1681752196000,0
...,...,...,...,...
24524,40.0,1678738917000,1682467275000,1
24525,40.0,1678738917000,1682478340000,1
24526,20.0,1678738917000,1682474490000,1
24527,40.0,1678740230000,1682488202000,0


In [20]:
from sklearn.metrics import mean_squared_error
mse=mean_squared_error(y_pred,y_test)
mse

,CON_SIZE,In_time,Out_time,L
0,40.0,1668216594000,1675732992000,0
1,40.0,1668216594000,1675366422000,0
2,20.0,1669124439000,1676005749000,0
3,20.0,1669124439000,1677572528000,0
4,20.0,1670173982000,1681752196000,0
...,...,...,...,...
24524,40.0,1678738917000,1682467275000,1
24525,40.0,1678738917000,1682478340000,1
24526,20.0,1678738917000,1682474490000,1
24527,40.0,1678740230000,1682488202000,0


In [21]:
df=pd.read_csv('Insert data.csv')
x = df['IN_TIME'][0]
x1 = datetime.strptime(x, '%d-%m-%y %H:%M:%S')
x1m=Milli(x1)
Out=[[40,x1m,1]]
x1=lr.predict(Out)
Outtime = Date_Time(int(x1))
print(Outtime)

2021-06-25 07:24:57


C:\ProgramData\Anaconda3\lib\site-packages\sklearn\base.py:450: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [22]:
df

,ID,IN_TIME,REF_ID,CON_NUM,CON_SIZE,STATUS
0,322,14-03-22 15:46:44,ALO11887,SPRE0057868,20,L
1,322,14-03-22 15:46:44,ALO11887,ANKJ1447873,20,L
2,322,14-03-22 15:46:44,ALO11887,ACLU3053768,20,L
3,326,17-04-22 04:41:42,DP02MAR10,BUDE0578181,40,L
4,326,17-04-22 04:41:42,DP02MAR10,UPDU1678037,20,E
...,...,...,...,...,...,...
345,554,20-03-22 14:48:13,ALO11980-98997-330244,IVIP5581378,20,L
346,555,20-03-22 14:48:13,ALO11980-98998-330245,XEFF1829672,20,L
347,556,20-03-22 14:48:13,ALO11980-98999-330246,BFIU2996554,20,L
348,557,20-03-22 14:48:13,ALO11980-99000-330247,RDCW7179253,20,L


In [19]:
print(df['Time_Diff'][0])
type(df['Time_Diff'][0])


86 days 23:53:18


pandas._libs.tslibs.timedeltas.Timedelta

In [20]:
q1=df['Time_Diff'].quantile(0.25)

In [21]:
print(q1)

0 days 05:21:09


KeyError: 'Out_Milli'

In [48]:
print(len(err))
err1=err

270


In [31]:
for i in range(len(err1)):
    err1[i]+=1

In [49]:
print(err)

[246, 931, 932, 933, 934, 1128, 1129, 1130, 1191, 1192, 1193, 1194, 1195, 1196, 1197, 1198, 1199, 1200, 1208, 1209, 1210, 1211, 1212, 1213, 1214, 1215, 1216, 1217, 1218, 1306, 1308, 1309, 1310, 1852, 1853, 1854, 1877, 3276, 3277, 3278, 3279, 3304, 3305, 3306, 3307, 3605, 3748, 3749, 3750, 3751, 3752, 3924, 3980, 4159, 4160, 4401, 4402, 4403, 4404, 4405, 4406, 4407, 4676, 4677, 4678, 5000, 5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008, 5014, 5068, 5081, 5120, 5231, 5232, 5374, 5400, 5401, 5402, 5403, 5590, 5591, 5598, 5601, 5941, 5968, 5969, 6753, 6754, 6755, 6756, 6757, 6758, 6759, 6760, 6761, 6762, 7091, 7136, 8110, 8111, 8112, 8113, 8114, 8115, 8934, 9390, 9580, 9581, 9582, 9583, 9584, 9741, 9747, 9916, 9920, 10319, 10447, 10448, 10449, 10450, 10451, 10452, 10453, 10599, 10600, 10601, 10602, 10603, 10604, 10650, 10651, 10652, 10653, 10718, 11006, 11007, 11008, 11009, 11010, 11011, 11012, 11013, 11037, 11053, 11436, 11538, 11813, 11814, 11893, 11894, 11895, 11896, 11897, 11898, 11899

In [50]:
df

,CON_SIZE,STATUS,In_time,Out_time
0,40.0,E,1668216594000,1675732992000
1,40.0,E,1668216594000,1675366422000
2,20.0,E,1669124439000,1676005749000
3,20.0,E,1669124439000,1677572528000
4,20.0,E,1670173982000,1681752196000
...,...,...,...,...
24799,40.0,L,1678738917000,1682467275000
24800,40.0,L,1678738917000,1682478340000
24801,20.0,L,1678738917000,1682474490000
24812,40.0,E,1678740230000,1682488202000


In [34]:
df.drop(labels=1)

,CON_SIZE,STATUS,In_time,Out_time
0,40.0,E,1668216594000,1675732992000
1,40.0,E,1668216594000,1675366422000
2,20.0,E,1669124439000,1676005749000
3,20.0,E,1669124439000,1677572528000
4,20.0,E,1670173982000,1681752196000
...,...,...,...,...
24798,40.0,L,1678738917000,1682555436000
24799,40.0,L,1678738917000,1682467275000
24800,40.0,L,1678738917000,1682478340000
24801,20.0,L,1678738917000,1682474490000


In [54]:
listre = err[::-1]
print(listre)
print(len(listre))
x = 99160
if x in listre:
    print('hy')

[24450, 24164, 24163, 24162, 24161, 24160, 24159, 24158, 24157, 24156, 24155, 24154, 24153, 23281, 23280, 23191, 23190, 22889, 22833, 22832, 22620, 22619, 22618, 22180, 22179, 22178, 22177, 22176, 22175, 22174, 22173, 22172, 22171, 22170, 22169, 22168, 21959, 21277, 21276, 21275, 21274, 21273, 21272, 20653, 20652, 20651, 20650, 20340, 20339, 20180, 20179, 20178, 20177, 20176, 20175, 20174, 20173, 19963, 19962, 19672, 19476, 19331, 19029, 19026, 19024, 19023, 19021, 19020, 18778, 18777, 18776, 18649, 18580, 17516, 17196, 17191, 15912, 15911, 15910, 15513, 15512, 15314, 15313, 15312, 14456, 14455, 14454, 14453, 14406, 14405, 14387, 14386, 13506, 13505, 13504, 13503, 13502, 13501, 13500, 12707, 12659, 12658, 12657, 12656, 12655, 12414, 12413, 12306, 12305, 11899, 11898, 11897, 11896, 11895, 11894, 11893, 11814, 11813, 11538, 11436, 11053, 11037, 11013, 11012, 11011, 11010, 11009, 11008, 11007, 11006, 10718, 10653, 10652, 10651, 10650, 10604, 10603, 10602, 10601, 10600, 10599, 10453, 10452

In [65]:
mer=[]
for i in range(df.shape[0]):
    if i in listre:
        try:
            df.drop(labels=i)
            print(i)
        except:
            mer.append(i)


In [62]:
print(listre)

[24450, 24164, 24163, 24162, 24161, 24160, 24159, 24158, 24157, 24156, 24155, 24154, 24153, 23281, 23280, 23191, 23190, 22889, 22833, 22832, 22620, 22619, 22618, 22180, 22179, 22178, 22177, 22176, 22175, 22174, 22173, 22172, 22171, 22170, 22169, 22168, 21959, 21277, 21276, 21275, 21274, 21273, 21272, 20653, 20652, 20651, 20650, 20340, 20339, 20180, 20179, 20178, 20177, 20176, 20175, 20174, 20173, 19963, 19962, 19672, 19476, 19331, 19029, 19026, 19024, 19023, 19021, 19020, 18778, 18777, 18776, 18649, 18580, 17516, 17196, 17191, 15912, 15911, 15910, 15513, 15512, 15314, 15313, 15312, 14456, 14455, 14454, 14453, 14406, 14405, 14387, 14386, 13506, 13505, 13504, 13503, 13502, 13501, 13500, 12707, 12659, 12658, 12657, 12656, 12655, 12414, 12413, 12306, 12305, 11899, 11898, 11897, 11896, 11895, 11894, 11893, 11814, 11813, 11538, 11436, 11053, 11037, 11013, 11012, 11011, 11010, 11009, 11008, 11007, 11006, 10718, 10653, 10652, 10651, 10650, 10604, 10603, 10602, 10601, 10600, 10599, 10453, 10452

In [66]:
print(mer)

[246, 931, 932, 933, 934, 1128, 1129, 1130, 1191, 1192, 1193, 1194, 1195, 1196, 1197, 1198, 1199, 1200, 1208, 1209, 1210, 1211, 1212, 1213, 1214, 1215, 1216, 1217, 1218, 1306, 1308, 1309, 1310, 1852, 1853, 1854, 1877, 3276, 3277, 3278, 3279, 3304, 3305, 3306, 3307, 3605, 3748, 3749, 3750, 3751, 3752, 3924, 3980, 4159, 4160, 4401, 4402, 4403, 4404, 4405, 4406, 4407, 4676, 4677, 4678, 5000, 5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008, 5014, 5068, 5081, 5120, 5231, 5232, 5374, 5400, 5401, 5402, 5403, 5590, 5591, 5598, 5601, 5941, 5968, 5969, 6753, 6754, 6755, 6756, 6757, 6758, 6759, 6760, 6761, 6762, 7091, 7136, 8110, 8111, 8112, 8113, 8114, 8115, 8934, 9390, 9580, 9581, 9582, 9583, 9584, 9741, 9747, 9916, 9920, 10319, 10447, 10448, 10449, 10450, 10451, 10452, 10453, 10599, 10600, 10601, 10602, 10603, 10604, 10650, 10651, 10652, 10653, 10718, 11006, 11007, 11008, 11009, 11010, 11011, 11012, 11013, 11037, 11053, 11436, 11538, 11813, 11814, 11893, 11894, 11895, 11896, 11897, 11898, 11899

In [68]:
df

,CON_SIZE,STATUS,In_time,Out_time
0,40.0,E,1668216594000,1675732992000
1,40.0,E,1668216594000,1675366422000
2,20.0,E,1669124439000,1676005749000
3,20.0,E,1669124439000,1677572528000
4,20.0,E,1670173982000,1681752196000
...,...,...,...,...
24799,40.0,L,1678738917000,1682467275000
24800,40.0,L,1678738917000,1682478340000
24801,20.0,L,1678738917000,1682474490000
24812,40.0,E,1678740230000,1682488202000


In [84]:
df['Out_time'][247]

1675525028000

In [85]:
df.reset_index()

,index,CON_SIZE,STATUS,In_time,Out_time
0,0,40.0,E,1668216594000,1675732992000
1,1,40.0,E,1668216594000,1675366422000
2,2,20.0,E,1669124439000,1676005749000
3,3,20.0,E,1669124439000,1677572528000
4,4,20.0,E,1670173982000,1681752196000
...,...,...,...,...,...
24524,24799,40.0,L,1678738917000,1682467275000
24525,24800,40.0,L,1678738917000,1682478340000
24526,24801,20.0,L,1678738917000,1682474490000
24527,24812,40.0,E,1678740230000,1682488202000


In [24]:
df

,ID,IN_TIME,REF_ID,CON_NUM,CON_SIZE,L
0,322,14-03-22 15:46:44,ALO11887,SPRE0057868,20,1
1,322,14-03-22 15:46:44,ALO11887,ANKJ1447873,20,1
2,322,14-03-22 15:46:44,ALO11887,ACLU3053768,20,1
3,326,17-04-22 04:41:42,DP02MAR10,BUDE0578181,40,1
4,326,17-04-22 04:41:42,DP02MAR10,UPDU1678037,20,0
...,...,...,...,...,...,...
345,554,20-03-22 14:48:13,ALO11980-98997-330244,IVIP5581378,20,1
346,555,20-03-22 14:48:13,ALO11980-98998-330245,XEFF1829672,20,1
347,556,20-03-22 14:48:13,ALO11980-98999-330246,BFIU2996554,20,1
348,557,20-03-22 14:48:13,ALO11980-99000-330247,RDCW7179253,20,1
